# Geoloc



## infos
Participants : Hélène DALON-DENEE, Dame DIENG, Célien GRIL, Nathan LEBRE

ligne 42368 : photo id 5464485473, correction -> les dates étaient décalée, le # de minutes (25) était collé au titre de la photo ("une lundi matin comme tout les autre ;-(") et le décalage était propagé.

In [ ]:
import numpy as np
import pandas as pd
import folium as fl
import hdbscan
import re
import matplotlib.pyplot as plt

In [ ]:
read_data = pd.read_csv("./flickr_data2.csv")
len(read_data)

Retirer les 142 lignes qui ont des valeurs non-nulles qui dépassent les colonnes attendues. (unnamed 16, y'a aussi 2 lignes avec des valeurs dans unnamed 18 mais elles sont comptabilisées dans les autres)

In [ ]:
wrong_data = read_data.dropna(how="all", subset=read_data.columns[[16, 18]])
fix_data = read_data.drop(index=wrong_data.index, axis=1)

toDropColumns = read_data.columns[[16, 17, 18]]
fix_data = fix_data.drop(toDropColumns, axis=1)

len(fix_data)

Retirer les tuples dupliquée (ignore id et tags pour enlever certains dupliqué quand un post est modifié et certains carousels, passe de 187544 à 175688 donc pas tant que ça, la majorité sont dupliqués point barre)

In [ ]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " tags"]), keep='last')
len(fix_data)

Retirer les carousels

In [ ]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " title"]))
len(fix_data)

Retirer les tags null pour le text pattern mining

In [ ]:
text_data = fix_data.dropna(how='all', subset=[" tags"])
len(text_data)

## Clustering

### HBDSCAN

In [ ]:
data2D = fix_data[[' lat', ' long']]
coords_rad = np.radians(data2D)
coords_rad.head()

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size = 100, min_samples = 50, metric= 'haversine')
labels = clusterer.fit_predict(coords_rad)
df = data2D.copy()
df["cluster"] = labels
pois = (
    df[df.cluster != -1]
    .groupby("cluster")
    .agg(
        lat_mean=(" lat", "mean"),
        lon_mean=(" long", "mean"),
        nb_photos=("cluster", "count")
    )
    .sort_values("nb_photos", ascending=False)
)

## Partie sur la carte.

In [ ]:
# Réinitialiser les indices pour éviter les problèmes
donnees_geoloc_reset = data2D.reset_index(drop=True)
labels_reset = labels.copy()

# Créer la carte centrée sur Lyon
map_hdbscan = fl.Map(
    location=[45.757778, 4.832222],
    zoom_start=12
)

# Couleurs pour les clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
          'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'gray']

# Ajouter les points avec les couleurs des clusters
for idx in range(len(donnees_geoloc_reset)):
    cluster_id = labels_reset[idx]
    row = donnees_geoloc_reset.iloc[idx]
    
    # Sauter le bruit (cluster -1)
    if cluster_id == -1:
        continue
    
    # Choisir la couleur selon le cluster
    color = colors[int(cluster_id) % len(colors)]
    
    # Ajouter un marqueur
    fl.CircleMarker(
        location=[row[' lat'], row[' long']],
        radius=3,
        popup=f'Cluster {cluster_id}',
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(map_hdbscan)

# Ajouter les points de bruit en gris transparent
for idx in range(len(donnees_geoloc_reset)):
    if labels_reset[idx] == -1:
        row = donnees_geoloc_reset.iloc[idx]
        fl.CircleMarker(
            location=[row[' lat'], row[' long']],
            radius=1,
            popup='Bruit',
            color='gray',
            fill=True,
            fillColor='gray',
            fillOpacity=0.2,
            weight=0.5
        ).add_to(map_hdbscan)

# Ajouter les centres des clusters (POIs)
for cluster_id, poi_row in pois.iterrows():
    fl.Marker(
        location=[poi_row['lat_mean'], poi_row['lon_mean']],
        popup=f"Cluster {cluster_id}<br>{int(poi_row['nb_photos'])} photos",
        icon=fl.Icon(color=colors[int(cluster_id) % len(colors)], icon='info-sign')
    ).add_to(map_hdbscan)

# Sauvegarder la carte
map_hdbscan.save('map_hdbscan.html')
print("Carte HDBSCAN sauvegardée dans 'map_hdbscan.html'")
print(f"\nRésumé des clusters HDBSCAN:")
print(pois)

# Text mining

In [ ]:
nona_df = text_data[' tags'].dropna()

#count = nona_df.apply(lambda x: len(re.findall(r'[^\W\d_]++', str(x)))).sum()
#print(f"Number of tags in the cleaned dataset: {count}")
print(nona_df)

In [ ]:
idf={}  # Inverse Document Frequency dictionary
total_docs = len(nona_df)
filtered_tags=["cospla","japa","feminicide","girl","hair","overwatch","tracer","aplusphoto","view","fiume","night","chaise","chair","iphone","streetphotography","rue","gens","nuit","francia","poste","river","notte","night"]

for tags in nona_df:
    unique_tags = set(re.findall(r'[^\W\d_]+', str(tags)))    
    unique_tags = {tag.lower() for tag in unique_tags if not any(bad in tag for bad in filtered_tags)}  # Normalize to lowercase
    for tag in unique_tags:
        idf[tag] = idf.get(tag, 0) + 1

for tag in idf:
    idf[tag] = total_docs / idf[tag]

print("IDF values for tags:")
for tag, value in idf.items():
    print(f"{tag}: {value}")

In [ ]:
mining_data = df.copy()
tags_data = fix_data[" tags"].str.split()
mining_data = mining_data.merge(tags_data, left_index=True, right_index=True)
mining_data = mining_data.dropna(subset=[" tags"])


In [ ]:
#print(mining_data[" tags"].get(1))

In [ ]:
#erreur pour le moment
tags_dict = {}
for row in mining_data.itertuples():
    clust = getattr(row, 'cluster')
    tags = getattr(row, '_4')
    if tags_dict.keys().__contains__(clust):
        tags_clust = tags_dict[clust]
        for tag in tags:
            print(tag)
            if tags_clust.keys().__contains__(tag):
                tags_clust[tag] += 1
            else:
                tags_clust[tag] = 1
        tags_dict[clust] = tags_clust
    else:
        tags_clust = {}
        for tag in tags:
            if tags_clust.keys().__contains__(tag):
                tags_clust[tag] += 1
            else:
                tags_clust[tag] = 1
        tags_dict[clust] = tags_clust


In [ ]:
tags_dict